In [21]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load cleaned datasets
train = pd.read_csv("train_cleaned.csv")
test = pd.read_csv("test_cleaned.csv")

# Separate target
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)

# One-hot encode categorical columns
X = pd.get_dummies(X)
X_test = pd.get_dummies(test)

# Align columns in test with train
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
import xgboost as xgb

# Convert to DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test)

params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "max_depth": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

evals = [(dtrain, "train"), (dval, "eval")]

model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50
)

# Predict
y_val_pred = model.predict(dval)
y_test_pred = model.predict(dtest)

[0]	train-rmse:0.37490	eval-rmse:0.41688
[50]	train-rmse:0.09699	eval-rmse:0.16085
[100]	train-rmse:0.06287	eval-rmse:0.14188
[150]	train-rmse:0.05069	eval-rmse:0.13842
[200]	train-rmse:0.04191	eval-rmse:0.13716
[250]	train-rmse:0.03533	eval-rmse:0.13709
[300]	train-rmse:0.03003	eval-rmse:0.13692
[350]	train-rmse:0.02568	eval-rmse:0.13680
[400]	train-rmse:0.02209	eval-rmse:0.13688
[411]	train-rmse:0.02146	eval-rmse:0.13700


In [27]:
import xgboost as xgb

# Convert test to DMatrix
dtest = xgb.DMatrix(X_test)

# Predict
y_test_pred = model.predict(dtest)

# Inverse log-transform
y_test_pred_final = np.expm1(y_test_pred)

# Save submission
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": y_test_pred_final
})

submission.to_csv("submission.csv", index=False)
print("Submission saved: submission.csv")

Submission saved: submission.csv
